# Sentiment Classification

Sentiment Classification is a **sequence classification** problem in NLP. Given a sentence or document, the model predicts a single sentiment label such as:

$$
\text{positive},\quad \text{negative}
$$

The key characteristic is that the input is a sequence containing multiple timesteps, while the output is a single label.

Therefore, Sentiment Classification is a typical:

$$
\boxed{
\text{Many-to-One Sequence Model}
}
$$

The general pipeline is:

$$
\boxed{
\text{Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Word IDs}
\rightarrow
\text{Word Embeddings}
\rightarrow
\text{RNN/LSTM/GRU}
\rightarrow
\text{Final Hidden State}
\rightarrow
\text{Classification}
}
$$

---

# 1. From Text to a Sequence Representation

Suppose the input sentence is:

> I really love this movie.

After tokenization:

$$
[\text{I},\text{really},\text{love},\text{this},\text{movie}]
$$

Each token is mapped to an embedding:

$$
e^{\langle1\rangle},
e^{\langle2\rangle},
\ldots,
e^{\langle T_x\rangle}
$$

where:

$$
e^{\langle t\rangle}\in\mathbb R^d
$$

These embeddings form the input sequence to the recurrent model.

For an RNN, the hidden state is updated recursively:

$$
\boxed{
a^{\langle t\rangle}
=
f\left(
a^{\langle t-1\rangle},
e^{\langle t\rangle}
\right)
}
$$

Therefore:

$$
a^{\langle1\rangle}
\rightarrow
a^{\langle2\rangle}
\rightarrow
\cdots
\rightarrow
a^{\langle T_x\rangle}
$$

The important idea is that each hidden state contains information accumulated from the sequence processed up to that timestep.

Conceptually:

$$
a^{\langle1\rangle}
=
f(e^{\langle1\rangle})
$$

$$
a^{\langle2\rangle}
=
f(e^{\langle1\rangle},e^{\langle2\rangle})
$$

and eventually:

$$
\boxed{
a^{\langle T_x\rangle}
=
\text{representation of the entire sequence}
}
$$

The final hidden state is therefore used as a compact representation of the sentence.

---

# 2. Why Is the Final Hidden State Used?

The task has only one target for the entire sentence:

$$
y\in\{0,1\}
$$

For example:

$$
y=1
\quad\text{for positive}
$$

and:

$$
y=0
\quad\text{for negative}
$$

Unlike sequence-to-sequence or sequence labeling tasks, we do not need one prediction for every timestep.

Instead:

$$
\boxed{
\text{Many Input Timesteps}
\rightarrow
\text{One Output}
}
$$

Hence the architecture is:

$$
e^{\langle1\rangle}
\rightarrow
a^{\langle1\rangle}
$$

$$
e^{\langle2\rangle}
\rightarrow
a^{\langle2\rangle}
$$

$$
\vdots
$$

$$
e^{\langle T_x\rangle}
\rightarrow
a^{\langle T_x\rangle}
\rightarrow
\hat y
$$

The model does **not** simply look at the last word.

Although the prediction is made from:

$$
a^{\langle T_x\rangle}
$$

this state was recursively built from the previous hidden states, which themselves depend on earlier words.

Thus:

$$
\boxed{
a^{\langle T_x\rangle}
=
f(e^{\langle1\rangle},\ldots,e^{\langle T_x\rangle})
}
$$

at the conceptual level.

So when we use the final hidden state, we are using the information accumulated from the entire sequence.

---

# 3. Output Layer and Sentiment Probability

Suppose:

$$
a^{\langle T_x\rangle}\in\mathbb R^{n_a}
$$

The output layer computes:

$$
\boxed{
z_y
=
W_ya^{\langle T_x\rangle}+b_y
}
$$

For binary sentiment classification:

$$
W_y\in\mathbb R^{1\times n_a}
$$

and:

$$
b_y\in\mathbb R
$$

The scalar $z_y$ is then passed through sigmoid:

$$
\boxed{
\hat y
=
\sigma(z_y)
=
\frac{1}{1+e^{-z_y}}
}
$$

Therefore:

$$
\hat y\in(0,1)
$$

and it can be interpreted as the model's estimated probability for the positive class.

For example:

$$
\hat y=0.95
$$

means the model strongly favors positive sentiment.

While:

$$
\hat y=0.05
$$

means the model strongly favors negative sentiment.

The complete output computation is:

$$
\boxed{
a^{\langle T_x\rangle}
\rightarrow
z_y
\rightarrow
\sigma(z_y)
\rightarrow
\hat y
}
$$

---

# 4. Loss Function

For binary sentiment classification, the standard loss is **Binary Cross-Entropy**:

$$
\boxed{
L
=
-\left[
y\log\hat y
+
(1-y)\log(1-\hat y)
\right]
}
$$

Consider the two possible labels.

If:

$$
y=1
$$

then:

$$
L=-\log\hat y
$$

So the model is rewarded for making:

$$
\hat y\rightarrow1
$$

For example:

$$
\hat y=0.95
\Rightarrow
L=-\log0.95
$$

which is small.

But if:

$$
\hat y=0.05
$$

then:

$$
L=-\log0.05
$$

which is large.

Similarly, if:

$$
y=0
$$

then:

$$
L=-\log(1-\hat y)
$$

and the model is rewarded for making:

$$
\hat y\rightarrow0
$$

Thus:

$$
\boxed{
\text{Correct and confident prediction}
\rightarrow
\text{small loss}
}
$$

while:

$$
\boxed{
\text{Wrong and confident prediction}
\rightarrow
\text{large loss}
}
$$

---

# 5. Forward Propagation

The complete forward pass for sentiment classification is:

$$
\boxed{
\text{Token IDs}
\rightarrow
\text{Embedding Lookup}
\rightarrow
e^{\langle1\rangle},\ldots,e^{\langle T_x\rangle}
}
$$

Then:

$$
\boxed{
e^{\langle1\rangle},\ldots,e^{\langle T_x\rangle}
\rightarrow
\text{RNN/LSTM/GRU}
\rightarrow
a^{\langle T_x\rangle}
}
$$

Then:

$$
\boxed{
a^{\langle T_x\rangle}
\rightarrow
W_ya^{\langle T_x\rangle}+b_y
\rightarrow
\sigma(\cdot)
\rightarrow
\hat y
}
$$

Finally:

$$
\boxed{
(y,\hat y)
\rightarrow
L
}
$$

So the entire forward computation is:

$$
\boxed{
\text{Text}
\rightarrow
\text{Embeddings}
\rightarrow
\text{Sequence Model}
\rightarrow
\text{Final State}
\rightarrow
\text{Probability}
\rightarrow
\text{Loss}
}
$$

---

# 6. Backpropagation Through Time

Although there is only one output, the gradient must still propagate through all timesteps.

Suppose:

$$
L=L(y,\hat y)
$$

The gradient first reaches:

$$
a^{\langle T_x\rangle}
$$

and then propagates backward through the recurrent chain:

$$
L
\rightarrow
a^{\langle T_x\rangle}
\rightarrow
a^{\langle T_x-1\rangle}
\rightarrow
\cdots
\rightarrow
a^{\langle1\rangle}
$$

This is **Backpropagation Through Time (BPTT)**.

The important consequence is that a word near the beginning of the sentence can still receive gradient information from the final sentiment prediction.

Conceptually:

$$
\boxed{
\text{Final Sentiment Error}
\rightarrow
\text{Backward Through Time}
\rightarrow
\text{Update Earlier Representations}
}
$$

Thus, the model can learn which words and relationships are useful for predicting sentiment.

---

# 7. Connection to Word Embeddings

The embedding layer is itself trainable.

Suppose the embedding matrix is:

$$
E\in\mathbb R^{d\times V}
$$

and a word with index $i$ has embedding:

$$
e_i=E_{:,i}
$$

Then:

$$
e_i
\rightarrow
\text{RNN/LSTM/GRU}
\rightarrow
\hat y
\rightarrow
L
$$

During backpropagation:

$$
L
\rightarrow
\frac{\partial L}{\partial e_i}
\rightarrow
\frac{\partial L}{\partial E}
$$

and:

$$
\boxed{
E
\leftarrow
E-\alpha\frac{\partial L}{\partial E}
}
$$

Therefore, when the embedding is trained jointly with sentiment classification, the model learns representations that are useful specifically for the sentiment task.

Alternatively, we can initialize $E$ from pretrained Word2Vec or GloVe vectors.

The workflow becomes:

$$
\boxed{
\text{Pretrained Embeddings}
\rightarrow
\text{Sequence Model}
\rightarrow
\text{Sentiment Classification}
}
$$

The embeddings can either be:

$$
\text{frozen}
$$

or:

$$
\text{fine-tuned}
$$

during training.

---

# 8. Why LSTM/GRU Can Be Useful

Consider a sentence such as:

> I initially thought the movie was boring, but after the second half I found it surprisingly enjoyable.

The sentiment depends on relationships across many words.

A vanilla RNN theoretically carries information through:

$$
a^{\langle1\rangle}
\rightarrow
a^{\langle2\rangle}
\rightarrow
\cdots
\rightarrow
a^{\langle T_x\rangle}
$$

but gradients must also propagate through this long chain during BPTT.

Therefore, long sequences can suffer from:

$$
\boxed{
\text{Vanishing / Exploding Gradients}
}
$$

which makes learning long-range dependencies difficult.

LSTM and GRU introduce gating mechanisms that provide better control over information flow.

Hence:

$$
\boxed{
\text{RNN}
\rightarrow
\text{basic recurrent sequence classifier}
}
$$

while:

$$
\boxed{
\text{LSTM/GRU}
\rightarrow
\text{better handling of long-term dependencies}
}
$$

This does not mean LSTM or GRU automatically understands sentiment; they simply provide a more effective recurrent mechanism for carrying relevant information through the sequence.

---

# 9. Tensor Shapes

Suppose:

$$
B=\text{batch size}
$$

$$
T=\text{sequence length}
$$

$$
V=\text{vocabulary size}
$$

$$
d=\text{embedding dimension}
$$

$$
n_a=\text{hidden dimension}
$$

Token IDs:

$$
\boxed{
X\in\mathbb R^{B\times T}
}
$$

Embedding lookup:

$$
\boxed{
X
\rightarrow
E(X)
\in
\mathbb R^{B\times T\times d}
}
$$

Sequence model:

$$
\boxed{
(B,T,d)
\rightarrow
(B,T,n_a)
}
$$

Final hidden state:

$$
\boxed{
a_T\in\mathbb R^{B\times n_a}
}
$$

Output layer:

$$
\boxed{
(B,n_a)
\rightarrow
(B,1)
}
$$

Sigmoid keeps the same shape:

$$
\boxed{
\hat y\in\mathbb R^{B\times1}
}
$$

So the whole pipeline is:

$$
\boxed{
(B,T)
\rightarrow
(B,T,d)
\rightarrow
(B,T,n_a)
\rightarrow
(B,n_a)
\rightarrow
(B,1)
}
$$

The sequence dimension $T$ is present while processing the sequence, but disappears when we take the final representation for sequence-level classification.

---

# 10. Mental Model

Sentiment Classification is fundamentally a **Many-to-One** sequence problem:

$$
\boxed{
x^{\langle1\rangle},
x^{\langle2\rangle},
\ldots,
x^{\langle T_x\rangle}
\rightarrow
a^{\langle T_x\rangle}
\rightarrow
\hat y
}
$$

The sequence model accumulates information:

$$
\boxed{
x^{\langle1\rangle}
\rightarrow
a^{\langle1\rangle}
}
$$

$$
\boxed{
a^{\langle1\rangle},x^{\langle2\rangle}
\rightarrow
a^{\langle2\rangle}
}
$$

$$
\boxed{
\cdots
}
$$

$$
\boxed{
a^{\langle T_x-1\rangle},x^{\langle T_x\rangle}
\rightarrow
a^{\langle T_x\rangle}
}
$$

Then:

$$
\boxed{
a^{\langle T_x\rangle}
\rightarrow
W_ya^{\langle T_x\rangle}+b_y
\rightarrow
\sigma(\cdot)
\rightarrow
\hat y
}
$$

and:

$$
\boxed{
\hat y
\rightarrow
L
\rightarrow
\text{BPTT}
\rightarrow
\text{Parameter Updates}
}
$$

The complete learning mechanism is:

$$
\boxed{
\text{Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Embedding}
\rightarrow
\text{RNN/LSTM/GRU}
\rightarrow
\text{Final Hidden State}
\rightarrow
\text{Binary Classification}
}
$$

The key point to remember is:

$$
\boxed{
\text{Many words}
\rightarrow
\text{one sequence representation}
\rightarrow
\text{one sentiment prediction}
}
$$

which is why Sentiment Classification is the canonical **Many-to-One** sequence model.

# Debiasing Word Embeddings

**Debiasing Word Embeddings** is the process of reducing unwanted social or stereotypical bias encoded in a pretrained word embedding space.

Because word embeddings are learned from large text corpora, they can absorb statistical patterns present in the training data. If the corpus contains biased associations, the learned vectors may encode those associations as geometric relationships.

For example, a model may learn an association such as:

$$
e_{\text{engineer}}
\text{ closer to }
e_{\text{man}}
$$

than to:

$$
e_{\text{woman}}
$$

even though gender should not determine whether someone is an engineer.

The core idea is:

$$
\boxed{
\text{Bias in Corpus}
\rightarrow
\text{Biased Statistical Patterns}
\rightarrow
\text{Biased Embedding Geometry}
}
$$

Debiasing attempts to modify the embedding space so that undesirable associations are reduced while useful semantic information is preserved.

---

# 1. Identifying the Bias Direction

The first step is to identify a direction in the embedding space corresponding to a particular type of bias.

Consider gender bias.

Suppose we have pairs such as:

$$
(\text{man},\text{woman})
$$

$$
(\text{boy},\text{girl})
$$

$$
(\text{father},\text{mother})
$$

A simple gender direction can be obtained from a difference vector:

$$
g
=
e_{\text{man}}
-
e_{\text{woman}}
$$

Similarly:

$$
g_1
=
e_{\text{boy}}
-
e_{\text{girl}}
$$

$$
g_2
=
e_{\text{father}}
-
e_{\text{mother}}
$$

These difference vectors tend to capture a common direction related to gender.

Using multiple word pairs gives a more stable estimate of the bias direction. Conceptually, we seek:

$$
\boxed{
g=\text{bias direction}
}
$$

This direction represents the component of the embedding space associated with the bias being studied.

To work with a unit direction, normalize it:

$$
\boxed{
\hat g=\frac{g}{\|g\|}
}
$$

---

# 2. Neutralize: Removing Bias from Neutral Words

Not every word should contain gender information.

Words such as:

$$
\text{engineer},\quad
\text{doctor},\quad
\text{teacher}
$$

are usually considered **neutral** with respect to gender.

If a neutral word has a large component along the gender direction, that component can represent an undesirable association.

Suppose the embedding is:

$$
e\in\mathbb R^d
$$

and the unit bias direction is:

$$
\hat g
$$

The projection of $e$ onto the bias direction is:

$$
\boxed{
\operatorname{proj}_{g}(e)
=
(e^T\hat g)\hat g
}
$$

This represents the component of the embedding along the bias direction.

We can remove it:

$$
\boxed{
e'
=
e-(e^T\hat g)\hat g
}
$$

This is called **neutralization**.

After neutralization:

$$
e'^T\hat g=0
$$

so the new vector has no component along the identified bias direction.

Conceptually:

$$
\boxed{
\text{Original Embedding}
=
\text{Bias Component}
+
\text{Remaining Information}
}
$$

and:

$$
\boxed{
\text{Neutralized Embedding}
=
\text{Original Embedding}
-
\text{Bias Component}
}
$$

The important point is that neutralization does **not** remove the entire vector. It removes only its component along the identified bias direction.

---

# 3. Geometric Interpretation of Neutralization

Suppose, for simplicity, that the embedding space is two-dimensional.

Let the horizontal axis represent the bias direction:

$$
g
$$

and let:

$$
e=
\begin{bmatrix}
a\\
b
\end{bmatrix}
$$

The component:

$$
a
$$

lies along the bias direction.

Neutralization removes this component:

$$
e'
=
\begin{bmatrix}
0\\
b
\end{bmatrix}
$$

Thus:

$$
\boxed{
\text{Neutralization}
=
\text{project the vector onto the subspace orthogonal to the bias direction}
}
$$

This geometric interpretation is the easiest way to understand:

$$
e'
=
e-(e^T\hat g)\hat g
$$

---

# 4. Why Should We Not Remove the Bias Direction from Every Word?

A crucial point is that **not all gender-related information is unwanted bias**.

Consider:

$$
\text{man}
\qquad\text{and}\qquad
\text{woman}
$$

or:

$$
\text{king}
\qquad\text{and}\qquad
\text{queen}
$$

Gender is part of their semantic distinction.

If we simply removed the entire gender component from every word, we could destroy meaningful information.

Therefore, debiasing needs to distinguish between:

$$
\boxed{
\text{Meaningful semantic information}
}
$$

and:

$$
\boxed{
\text{Unwanted biased association}
}
$$

This motivates the second operation: **equalization**.

---

# 5. Equalize: Making Related Word Pairs More Symmetric

Some words naturally occur in gendered pairs:

$$
\{\text{man},\text{woman}\}
$$

$$
\{\text{king},\text{queen}\}
$$

These are called **equality sets** in the simplified debiasing framework.

The goal is not to make the embeddings identical.

Instead, we want them to be more symmetric with respect to the bias direction while preserving their meaningful distinction.

Suppose we have a pair:

$$
e_1,\qquad e_2
$$

such as:

$$
e_{\text{man}},
\qquad
e_{\text{woman}}
$$

First compute their mean:

$$
\mu
=
\frac{e_1+e_2}{2}
$$

Then decompose the mean into:

$$
\mu=\mu_B+\mu_\perp
$$

where:

- $\mu_B$ is the component along the bias direction;
- $\mu_\perp$ is the component orthogonal to the bias direction.

The idea of equalization is to preserve the common semantic component while making the two words symmetric around the neutral subspace.

Conceptually:

$$
\boxed{
\text{Equalize}
=
\text{Preserve meaningful difference}
+
\text{Reduce asymmetric bias}
}
$$

Thus we should not think of equalization as:

$$
e_1=e_2
$$

Instead, we want:

$$
\boxed{
e_1'
\text{ and }
e_2'
\text{ to be symmetric around the neutral component}
}
$$

---

# 6. Neutralize vs. Equalize

This distinction is essential.

### Neutralization

Applied to words that should be neutral with respect to the bias.

Example:

$$
\text{engineer}
$$

The objective is:

$$
\boxed{
\text{Remove the unwanted projection onto the bias direction}
}
$$

using:

$$
e'
=
e-(e^T\hat g)\hat g
$$

### Equalization

Applied to words that legitimately form a semantic pair associated with the bias.

Examples:

$$
\text{man/woman}
$$

$$
\text{king/queen}
$$

The objective is:

$$
\boxed{
\text{Preserve meaningful distinction}
+
\text{make the representation more symmetric}
}
$$

Therefore:

$$
\boxed{
\text{Neutralize}
\rightarrow
\text{remove unwanted bias from neutral words}
}
$$

while:

$$
\boxed{
\text{Equalize}
\rightarrow
\text{reduce asymmetric bias in meaningful paired words}
}
$$

---

# 7. A Concrete Example

Suppose the embedding space contains a gender direction:

$$
g
$$

and:

$$
e_{\text{engineer}}
$$

has a strong projection onto $g$:

$$
e_{\text{engineer}}^T\hat g\gg0
$$

This means the embedding has a strong component in the identified gender direction.

We neutralize it:

$$
\boxed{
e'_{\text{engineer}}
=
e_{\text{engineer}}
-
(e_{\text{engineer}}^T\hat g)\hat g
}
$$

Now:

$$
e'_{\text{engineer}}{}^T\hat g=0
$$

The representation of `engineer` is therefore no longer positioned along that gender direction.

However, for:

$$
\text{man}
\quad\text{and}\quad
\text{woman}
$$

we should not simply remove the gender direction because the distinction itself is meaningful.

Instead, equalization tries to place their embeddings symmetrically around the neutral component.

---

# 8. Complete Debiasing Procedure

The simplified procedure can be summarized as:

$$
\boxed{
\text{Pretrained Word Embeddings}
\rightarrow
\text{Identify Bias Direction}
\rightarrow
\text{Neutralize Neutral Words}
\rightarrow
\text{Equalize Relevant Word Pairs}
}
$$

More explicitly:

$$
E
\rightarrow
g
$$

then for a neutral word:

$$
e
\rightarrow
e-(e^T\hat g)\hat g
$$

and for equality sets:

$$
\text{adjust vectors toward symmetric representations}
$$

The final result is:

$$
\boxed{
E_{\text{debiased}}
}
$$

---

# 9. Why Debiasing Is Not Perfect

A very important limitation is that real-world bias does not necessarily lie in a single clean direction.

The simplified model assumes:

$$
\boxed{
\text{Bias}
\approx
\text{one identifiable direction}
}
$$

But in real embedding spaces:

$$
\text{Bias}
\rightarrow
\text{multiple directions}
+
\text{complex interactions}
$$

Therefore, removing one direction cannot guarantee that all bias has been eliminated.

Furthermore, even if the embedding itself becomes less biased, downstream models can still learn biased associations from:

- other dimensions of the embedding;
- downstream training data;
- model architecture;
- other features.

Therefore:

$$
\boxed{
\text{Debiased Embedding}
\neq
\text{Completely Unbiased NLP System}
}
$$

Debiasing should be understood as a method for **reducing specific undesirable associations**, not as a guarantee that all forms of bias disappear.

---

# 10. Connection to Word2Vec and GloVe

Debiasing is not another word-embedding learning architecture like:

$$
\text{Word2Vec}
$$

or:

$$
\text{GloVe}
$$

Instead, it is a **post-processing procedure** that can be applied after embeddings have already been learned.

For example:

$$
E_{\text{Word2Vec}}
\rightarrow
\text{Debiasing}
\rightarrow
E'_{\text{Word2Vec}}
$$

or:

$$
E_{\text{GloVe}}
\rightarrow
\text{Debiasing}
\rightarrow
E'_{\text{GloVe}}
$$

The underlying idea is therefore:

$$
\boxed{
\text{Learn Embedding}
\rightarrow
\text{Inspect Embedding Geometry}
\rightarrow
\text{Modify Unwanted Bias}
}
$$

---

# Final Mental Model

The entire topic can be reduced to three ideas.

First, identify a bias direction:

$$
\boxed{
g=\text{bias direction}
}
$$

Second, for neutral words, remove the projection onto that direction:

$$
\boxed{
e'
=
e-(e^T\hat g)\hat g
}
$$

Third, for meaningful paired words, use equalization rather than simply deleting the bias-related information:

$$
\boxed{
\text{Equalize}
=
\text{preserve semantic distinction}
+
\text{reduce asymmetric bias}
}
$$

Therefore:

$$
\boxed{
\text{Biased Embedding Space}
\rightarrow
\text{Bias Direction}
\rightarrow
\text{Neutralize}
\rightarrow
\text{Equalize}
\rightarrow
\text{Reduced Bias}
}
$$

The deepest idea is:

$$
\boxed{
\text{Debiasing should remove unwanted associations, not destroy meaningful semantics}
}
$$

And the most important distinction to remember is:

$$
\boxed{
\text{Neutralize}
\rightarrow
\text{remove bias from neutral words}
}
$$

$$
\boxed{
\text{Equalize}
\rightarrow
\text{make meaningful paired words more symmetric}
}
$$